In [ ]:
# ==========================
# Cell 1 - Initialize
# ==========================

from urllib.parse import unquote
import json

import psycopg
from azure.storage.blob import BlobServiceClient

# ========= Azure Blob =========

CONNECT_STR = ""   # TODO: thay bằng connection string
CONTAINER_NAME = "raw-documents"

blob_service_client = BlobServiceClient.from_connection_string(CONNECT_STR)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# ========= PostgreSQL =========

DB_CONFIG = {
    "host": "dbuatnexssi.postgres.database.azure.com",       # TODO
    "port": 5432,
    "dbname": "postgres",          # TODO
    "user": "thaitn36514",        # TODO
    "password": "Sacombank@123"     # TODO
}

conn = psycopg.connect(**DB_CONFIG)

# ========= Test =========

print("✅ Connected to Azure Blob Storage")
print(f"Container: {CONTAINER_NAME}")

with conn.cursor() as cur:
    cur.execute("SELECT version();")
    version = cur.fetchone()[0]

print("✅ Connected to PostgreSQL")
print(version)

In [ ]:
# ==========================
# Cell 2 - Read metadata from Blob
# ==========================

from urllib.parse import unquote

def decode_sharepoint_metadata(metadata: dict) -> dict:
    """
    Decode URL-encoded values trong metadata SharePoint.
    """
    decoded = {}

    for key, value in metadata.items():
        if isinstance(value, str):
            decoded[key] = unquote(value)
        else:
            decoded[key] = value

    return decoded


documents = []

for blob in container_client.list_blobs():

    blob_client = container_client.get_blob_client(blob.name)
    properties = blob_client.get_blob_properties()

    metadata = decode_sharepoint_metadata(properties.metadata)

    document = {
        "blob_name": blob.name,
        "file_size_bytes": properties.size,
        "last_modified": properties.last_modified.isoformat(),
        "metadata": metadata
    }

    documents.append(document)

print(f"✅ Total documents: {len(documents)}")

if documents:
    print("\n===== Sample Document =====")
    print(json.dumps(documents[0], ensure_ascii=False, indent=4))


In [ ]:
# # ==========================
# # Cell 3 - Sync PostgreSQL Schema
# # ==========================

# import re

# TABLE_NAME = "vblq_metadata"

# system_columns = {
#     "blob_name": "TEXT PRIMARY KEY",
#     "blob_file_size_bytes": "BIGINT",
#     "blob_last_modified": "TIMESTAMPTZ"
# }

# rename_map = {
#     "file_size_bytes": "sharepoint_file_size_bytes",
#     "last_modified": "sharepoint_last_modified"
# }


# def normalize_column_name(name: str) -> str:
#     name = rename_map.get(name, name)
#     name = name.replace(" ", "_")
#     name = re.sub(r"[^0-9a-zA-Z_]", "_", name)
#     name = re.sub(r"_+", "_", name)
#     return name.lower()


# # =====================================================
# # Thu thập metadata fields
# # =====================================================

# metadata_fields = set()

# for doc in documents:
#     metadata_fields.update(doc["metadata"].keys())

# metadata_fields = sorted(metadata_fields)

# column_mapping = {}

# # =====================================================
# # Kiểm tra table tồn tại chưa
# # =====================================================

# with conn.cursor() as cur:

#     cur.execute("""
#         SELECT EXISTS (
#             SELECT 1
#             FROM information_schema.tables
#             WHERE table_name = %s
#         );
#     """, (TABLE_NAME,))

#     table_exists = cur.fetchone()[0]

# # =====================================================
# # CREATE TABLE
# # =====================================================

# if not table_exists:

#     create_columns = []

#     for col, dtype in system_columns.items():
#         create_columns.append(f"{col} {dtype}")

#     for field in metadata_fields:

#         column_name = normalize_column_name(field)

#         column_mapping[field] = column_name

#         create_columns.append(f"{column_name} TEXT")

#     create_columns.extend([
#         "created_at TIMESTAMPTZ DEFAULT NOW()",
#         "updated_at TIMESTAMPTZ DEFAULT NOW()"
#     ])

#     create_sql = f"""
#     CREATE TABLE {TABLE_NAME}
#     (
#         {', '.join(create_columns)}
#     );
#     """

#     with conn.cursor() as cur:
#         cur.execute(create_sql)

#     conn.commit()

#     print(f"✅ Table '{TABLE_NAME}' created.")

# # =====================================================
# # ALTER TABLE
# # =====================================================

# else:

#     print(f"ℹ️ Table '{TABLE_NAME}' already exists.")

#     with conn.cursor() as cur:

#         cur.execute("""
#             SELECT column_name
#             FROM information_schema.columns
#             WHERE table_name = %s
#         """, (TABLE_NAME,))

#         existing_columns = {
#             row[0]
#             for row in cur.fetchall()
#         }

#     # system columns
#     for col in system_columns:
#         existing_columns.add(col)

#     added = 0

#     with conn.cursor() as cur:

#         for field in metadata_fields:

#             column_name = normalize_column_name(field)

#             column_mapping[field] = column_name

#             if column_name not in existing_columns:

#                 sql = f"""
#                 ALTER TABLE {TABLE_NAME}
#                 ADD COLUMN {column_name} TEXT;
#                 """

#                 cur.execute(sql)

#                 print(f"➕ Added column: {column_name}")

#                 added += 1

#     conn.commit()

#     if added == 0:
#         print("✅ Schema is already up-to-date.")
#     else:
#         print(f"✅ Added {added} new columns.")

ℹ️ Table 'vblq_metadata' already exists.
✅ Schema is already up-to-date.


In [26]:
# ==========================
# Cell 3 - Sync PostgreSQL Schema
# ==========================

import re

TABLE_NAME = "vblq_metadata"
SCHEMA_NAME = "public"


# ==========================
# System columns
# ==========================

system_columns = {
    "blob_name": "TEXT PRIMARY KEY",
    "blob_file_size_bytes": "BIGINT",
    "blob_last_modified": "TIMESTAMPTZ"
}


# tránh trùng tên
rename_map = {
    "file_size_bytes": "sharepoint_file_size_bytes",
    "last_modified": "sharepoint_last_modified"
}


def normalize_column_name(name: str) -> str:
    """
    Chuẩn hóa tên column PostgreSQL
    """

    name = rename_map.get(name, name)

    name = name.replace(" ", "_")

    name = re.sub(
        r"[^0-9a-zA-Z_]",
        "_",
        name
    )

    name = re.sub(
        r"_+",
        "_",
        name
    )

    return name.lower()


try:

    # ==========================
    # Collect metadata fields
    # ==========================

    metadata_fields = set()

    for doc in documents:
        metadata_fields.update(
            doc["metadata"].keys()
        )


    metadata_fields = sorted(metadata_fields)


    # mapping:
    # SharePoint field -> DB column
    column_mapping = {}

    for field in metadata_fields:
        column_mapping[field] = normalize_column_name(field)


    print(
        f"Detected metadata fields: {len(metadata_fields)}"
    )


    # ==========================
    # Check table exists
    # ==========================

    with conn.cursor() as cur:

        cur.execute(
            """
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_schema = %s
                AND table_name = %s
            );
            """,
            (
                SCHEMA_NAME,
                TABLE_NAME
            )
        )

        table_exists = cur.fetchone()[0]


    # ==========================
    # CREATE TABLE
    # ==========================

    if not table_exists:

        columns = []


        for col, dtype in system_columns.items():
            columns.append(
                f"{col} {dtype}"
            )


        for field, column_name in column_mapping.items():

            columns.append(
                f"{column_name} TEXT"
            )


        columns.extend(
            [
                "created_at TIMESTAMPTZ DEFAULT NOW()",
                "updated_at TIMESTAMPTZ DEFAULT NOW()"
            ]
        )


        create_sql = f"""
        CREATE TABLE {SCHEMA_NAME}.{TABLE_NAME}
        (
            {", ".join(columns)}
        );
        """


        with conn.cursor() as cur:
            cur.execute(create_sql)


        conn.commit()

        print(
            f"✅ Created table {TABLE_NAME}"
        )


    # ==========================
    # ALTER TABLE
    # ==========================

    else:

        print(
            f"ℹ️ Table {TABLE_NAME} already exists"
        )


        with conn.cursor() as cur:

            cur.execute(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = %s
                AND table_name = %s;
                """,
                (
                    SCHEMA_NAME,
                    TABLE_NAME
                )
            )

            existing_columns = {
                row[0]
                for row in cur.fetchall()
            }


        added_columns = 0


        with conn.cursor() as cur:

            for field, column_name in column_mapping.items():


                if column_name not in existing_columns:

                    alter_sql = f"""
                    ALTER TABLE {SCHEMA_NAME}.{TABLE_NAME}
                    ADD COLUMN {column_name} TEXT;
                    """

                    cur.execute(alter_sql)

                    print(
                        f"➕ Added column: {column_name}"
                    )

                    added_columns += 1


        conn.commit()


        if added_columns == 0:

            print(
                "✅ Schema already up-to-date"
            )

        else:

            print(
                f"✅ Added {added_columns} columns"
            )


except Exception as e:

    conn.rollback()

    print(
        "❌ Schema sync failed:"
    )

    print(e)

    raise


print("\n========== Column Mapping ==========")

for source, target in column_mapping.items():

    print(
        f"{source:35} -> {target}"
    )

Detected metadata fields: 21
ℹ️ Table vblq_metadata already exists
✅ Schema already up-to-date

========== Column Mapping ==========
Linh_vuc                            -> linh_vuc
Loai_van_ban                        -> loai_van_ban
Ngay_ban_hanh                       -> ngay_ban_hanh
Ngay_het_hieu_luc                   -> ngay_het_hieu_luc
Ngay_hieu_luc                       -> ngay_hieu_luc
So_quyet_dinh                       -> so_quyet_dinh
So_thu_tu                           -> so_thu_tu
Tinh_trang                          -> tinh_trang
Trich_yeu                           -> trich_yeu
Van_ban_bi_thay_the                 -> van_ban_bi_thay_the
etag                                -> etag
file_id                             -> file_id
file_name                           -> file_name
file_size_bytes                     -> sharepoint_file_size_bytes
last_modified                       -> sharepoint_last_modified
server_relative_url                 -> server_relative_url
sharepoint_item

In [27]:
# ==========================
# Cell 4 - Sync metadata
# ==========================

db_columns = list(system_columns.keys()) + list(column_mapping.values())
update_columns = [c for c in db_columns if c != "blob_name"]

insert_sql = f"""
INSERT INTO {TABLE_NAME}
(
    {", ".join(db_columns)}
)
VALUES
(
    {", ".join(["%s"] * len(db_columns))}
)
ON CONFLICT (blob_name)
DO UPDATE
SET
    {", ".join([f"{c} = EXCLUDED.{c}" for c in update_columns])},
    updated_at = NOW()
WHERE
    {" OR ".join([
        f"{TABLE_NAME}.{c} IS DISTINCT FROM EXCLUDED.{c}"
        for c in update_columns
    ])};
"""

processed = 0
applied = 0
skipped = 0

with conn.cursor() as cur:

    for doc in documents:

        metadata = doc["metadata"]

        values = [
            doc["blob_name"],
            doc["file_size_bytes"],
            doc["last_modified"],
        ]

        # Thêm toàn bộ metadata theo đúng mapping
        for source_field in column_mapping:
            values.append(metadata.get(source_field))

        cur.execute(insert_sql, values)

        processed += 1

        # rowcount:
        # 1 = INSERT hoặc UPDATE
        # 0 = Conflict nhưng dữ liệu không đổi (SKIP)
        if cur.rowcount == 1:
            applied += 1
        else:
            skipped += 1

conn.commit()

print("=" * 60)
print("Metadata synchronization completed")
print("=" * 60)
print(f"Processed : {processed}")
print(f"Applied   : {applied}")
print(f"Skipped   : {skipped}")
print("=" * 60)

Metadata synchronization completed
Processed : 2
Applied   : 0
Skipped   : 2


In [ ]:
conn.rollback()

print("✅ Transaction rolled back.")